# Exercise
Download heart disease dataset heart.csv in Exercise folder and do following, (credits of dataset: https://www.kaggle.com/fedesoriano/heart-failure-prediction)

1. Load heart disease dataset in pandas dataframe
2. Remove outliers using Z score. Usual guideline is to remove anything that has Z score > 3 formula or Z score < -3
3. Convert text columns to numbers using label encoding and one hot encoding
4. Apply scaling
5. Build a classification model using support vector machine. Use standalone model as well as Bagging model and check if you see any difference in the performance.
6. Now use decision tree classifier. Use standalone model as well as Bagging and check if you notice any difference in performance
7. Comparing performance of svm and decision tree classifier figure out where it makes most sense to use bagging and why. Use internet to figure out in what conditions bagging works the best.

In [277]:
import pandas as pd
df = pd.read_csv("heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [278]:
df.shape

(918, 12)

In [279]:
count = len(df.columns)

In [280]:
from pandas.api.types import is_numeric_dtype

for i in range(count):
    col = df.columns[i]
    if is_numeric_dtype(df[col]):
        df = df[df[col] <= (3 * df[col].std() + df[col].mean())]

In [281]:
df.shape

(902, 12)

In [282]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [283]:
print(df.RestingECG.unique())
print(df.ExerciseAngina.unique())
print(df.ST_Slope.unique())

<StringArray>
['Normal', 'ST', 'LVH']
Length: 3, dtype: str
<StringArray>
['N', 'Y']
Length: 2, dtype: str
<StringArray>
['Up', 'Flat', 'Down']
Length: 3, dtype: str


In [284]:

df = df.replace({
    'RestingECG' : {
        'Normal' : 1,
        'ST'    : 2,
        'LVH'   : 3
    },
    'ExerciseAngina': {
        'N' : 1,
        'Y' : 2,
    },
    'ST_Slope' : {
        'Up'    : 1,
        'Flat'  : 2,
        'Down'  : 3
    }

})

In [285]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,1,172,1,0.0,1,0
1,49,F,NAP,160,180,0,1,156,1,1.0,2,1
2,37,M,ATA,130,283,0,2,98,1,0.0,1,0
3,48,F,ASY,138,214,0,1,108,2,1.5,2,1
4,54,M,NAP,150,195,0,1,122,1,0.0,1,0


In [286]:
df = pd.get_dummies(df, drop_first= True, dtype = int)

In [287]:
df.shape

(902, 16)

In [288]:
df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_2,RestingECG_3,ExerciseAngina_2,ST_Slope_2,ST_Slope_3
0,40,140,289,0,172,0.0,0,1,1,0,0,0,0,0,0,0
1,49,160,180,0,156,1.0,1,0,0,1,0,0,0,0,1,0
2,37,130,283,0,98,0.0,0,1,1,0,0,1,0,0,0,0
3,48,138,214,0,108,1.5,1,0,0,0,0,0,0,1,1,0
4,54,150,195,0,122,0.0,0,1,0,1,0,0,0,0,0,0


In [291]:
X = df.drop("HeartDisease", axis = 1)
y = df.HeartDisease

In [292]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[-1.42896269,  0.46089071,  0.85238015, ..., -0.82065181,
        -1.00221976, -0.2597222 ],
       [-0.47545956,  1.5925728 , -0.16132855, ..., -0.82065181,
         0.99778516, -0.2597222 ],
       [-1.74679706, -0.10495034,  0.79657967, ..., -0.82065181,
        -1.00221976, -0.2597222 ],
       ...,
       [ 0.37209878, -0.10495034, -0.61703246, ...,  1.21854359,
         0.99778516, -0.2597222 ],
       [ 0.37209878, -0.10495034,  0.35947592, ..., -0.82065181,
         0.99778516, -0.2597222 ],
       [-1.64085227,  0.3477225 , -0.20782894, ..., -0.82065181,
        -1.00221976, -0.2597222 ]], shape=(902, 15))

In [293]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, stratify=y, test_size= 0.2,random_state=10)

In [294]:
X_train.shape

(721, 15)

In [295]:
X_test.shape

(181, 15)

In [296]:
y_train.value_counts()

HeartDisease
1    396
0    325
Name: count, dtype: int64

In [297]:
325/396

0.8207070707070707

In [298]:
y_test.value_counts()

HeartDisease
1    99
0    82
Name: count, dtype: int64

In [299]:
82/99

0.8282828282828283

# Train using stand alone model

In [301]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

scores = cross_val_score(DecisionTreeClassifier(), X, y, cv=5)
scores


array([0.83425414, 0.74033149, 0.75      , 0.71666667, 0.68333333])

In [302]:
scores.mean()

np.float64(0.7449171270718231)

# Train using Bagging

In [304]:
from sklearn.ensemble import BaggingClassifier

from sklearn.tree import DecisionTreeClassifier

bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=0
)

bag_model.fit(X_train, y_train)

bag_model.oob_score_

0.8502080443828016

In [305]:
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=0
)

scores = cross_val_score(bag_model,X,y,cv = 5)
scores.mean()

np.float64(0.7948373235113566)

# Train a model Using standalone SVM and using bagging

In [306]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

scores = cross_val_score(SVC(), X, y, cv=5)
scores.mean()

np.float64(0.6895273173726213)

In [308]:
from sklearn.ensemble import BaggingClassifier

bag_model = BaggingClassifier(estimator=SVC(), n_estimators=100, max_samples=0.8, random_state=0)
scores = cross_val_score(bag_model, X, y, cv=5)
scores.mean()

np.float64(0.6850767341927564)